# Explore the AutoStrat → EvoMachine pipeline

This notebook sends one natural-language request to an OpenAI-compatible model, validates and semantically verifies the generated DSL, wraps it as an EvoMachine `AbstractStrategy`, and previews the resulting `AutomatonCommand` lists.

It does **not** start the automaton or communicate with microscope hardware.

In [1]:
from getpass import getpass
import os
from pathlib import Path

from autostrat import StrategyPipeline, load_domain_pack
from autostrat.generation import GeneratorConfig, PromptRecipe
from autostrat.verification import SemanticVerifierConfig

from evomachine.coordinates import Coordinate
from evomachine.image_processing_config import ImageProcessorConfigFactory
from evomachine.strategy_generation import (
    MicroscopyCommandAdapter,
    MicroscopyObservationProvider,
    StrategyGenerationService,
)
from evomachine.types import LEDType

/Users/liammetcalf/Code/workspace/evomachine/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:bfio.init:VERSION = 2.3.0

INFO:bfio.init:The bioformats_package.jar is not present.Can only use Python backend for reading/writing images.


In [2]:
def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'evomachine').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from inside the EvoMachine repository.')


repository_root = find_repository_root()
domain = load_domain_pack(repository_root / 'evomachine/domain_packs/microscopy')
print(f'Repository: {repository_root}')
print(f'Domain: {domain.metadata.id} {domain.metadata.version}')
print(f'Commands: {list(domain.commands)}')
print(f'Observations: {list(domain.observations)}')

Repository: /Users/liammetcalf/Code/workspace/evomachine
Domain: microscopy 0.1.0
Commands: ['move_fov', 'image', 'wait']
Observations: ['current_fov_id', 'step_count']


## Model connection

The defaults below target Robin's OpenAI-compatible endpoint. The key is read from `OPENAI_API_KEY`, or requested without displaying it. Nothing is written to the repository.

In [3]:
base_url = os.getenv('OPENAI_BASE_URL', 'https://robin-office-2.tail32bb7.ts.net/v1')
model_id = os.getenv('AUTOSTRAT_MODEL_ID', 'qwen3.6')
model = model_id if ':' in model_id else f'openai-chat:{model_id}'

os.environ['OPENAI_BASE_URL'] = base_url
if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI-compatible API key: ')

print(f'Endpoint: {base_url}')
print(f'Pydantic AI model: {model}')

Endpoint: https://robin-office-2.tail32bb7.ts.net/v1
Pydantic AI model: openai-chat:qwen3.6


## Editable experiment settings

In [22]:
user_request = (
"""


During initialisation:

- Move to the first field of view.
- Capture an image using an exposure of 7 ms and the 385 nm LED.
- Wait for 0.25 s.

During every step, first inspect the number of previously completed steps.

If at least 12 steps have already been completed, terminate immediately. Do not move, image, or wait in that branch.

Otherwise, use the following behaviour:

If fewer than 3 steps have been completed:

- If the current field-of-view identifier is 0:
  - Capture an image using an exposure of 11 ms and the 450 nm LED.
  - Wait for 0.5 s.
  - Move to the next field of view.
- Otherwise:
  - Capture an image using an exposure of 13 ms and the 515 nm LED.
  - Wait for 0.75 s.
  - Move to the next field of view.

Otherwise, if fewer than 6 steps have been completed:

- Move to the next field of view.
- If the current field-of-view identifier is 0:
  - Capture an image using an exposure of 17 ms and the 565 nm LED.
  - Wait for 1 s.
- Otherwise:
  - If the current field-of-view identifier is greater than 2:
    - Capture an image using an exposure of 19 ms and the 645 nm LED.
    - Wait for 1.25 s.
  - Otherwise:
    - Capture an image using an exposure of 23 ms and the 385 nm LED.
    - Wait for 1.5 s.

Otherwise, if fewer than 9 steps have been completed:

- If the current field-of-view identifier is not 0:
  - Move to the next field of view.
  - Capture an image using an exposure of 29 ms and the 450 nm LED.
  - Wait for 2 s.
- Otherwise:
  - Capture an image using an exposure of 31 ms and the 515 nm LED.
  - Wait for 2.5 s.
  - Move to the next field of view.

Otherwise:

- If fewer than 11 steps have been completed:
  - Move to the next field of view.
  - If the current field-of-view identifier is at least 2:
    - Capture an image using an exposure of 37 ms and the 565 nm LED.
    - Wait for 3 s.
  - Otherwise:
    - Capture an image using an exposure of 41 ms and the 645 nm LED.
    - Wait for 3.5 s.
- Otherwise:
  - If the current field-of-view identifier is 0:
    - Capture an image using an exposure of 43 ms and the 385 nm LED.
    - Wait for 4 s.
  - Otherwise:
    - Capture an image using an exposure of 47 ms and the 450 nm LED.
    - Wait for 4.5 s.
  - Move to the next field of view after completing whichever imaging branch applies.

During finalisation:

- Capture an image using an exposure of 50 ms and the 645 nm LED.
- Wait for 1 s.
- Move to the first field of view.


""")

few_shot_count = 2
validation_retries = 2
semantic_output_retries = 2
semantic_revisions = 2

# These are application-owned imaging policies, not generated DSL arguments.
image_brightness = 10
segment_images = False
save_images = False

print(user_request)




During initialisation:

- Move to the first field of view.
- Capture an image using an exposure of 7 ms and the 385 nm LED.
- Wait for 0.25 s.

During every step, first inspect the number of previously completed steps.

If at least 12 steps have already been completed, terminate immediately. Do not move, image, or wait in that branch.

Otherwise, use the following behaviour:

If fewer than 3 steps have been completed:

- If the current field-of-view identifier is 0:
  - Capture an image using an exposure of 11 ms and the 450 nm LED.
  - Wait for 0.5 s.
  - Move to the next field of view.
- Otherwise:
  - Capture an image using an exposure of 13 ms and the 515 nm LED.
  - Wait for 0.75 s.
  - Move to the next field of view.

Otherwise, if fewer than 6 steps have been completed:

- Move to the next field of view.
- If the current field-of-view identifier is 0:
  - Capture an image using an exposure of 17 ms and the 565 nm LED.
  - Wait for 1 s.
- Otherwise:
  - If the current field-of

## Run generation, deterministic validation and semantic verification

In [23]:
pipeline = StrategyPipeline(
    domain,
    generator_config=GeneratorConfig(
        model=model,
        validation_retries=validation_retries,
    ),
    verifier_config=SemanticVerifierConfig(
        model=model,
        output_retries=semantic_output_retries,
    ),
    prompt_recipe=PromptRecipe(
        name='evomachine-notebook',
        few_shot_count=few_shot_count,
    ),
    semantic_revisions=semantic_revisions,
)

cfg = ImageProcessorConfigFactory.default_config(
    channels=[LEDType.LED_450_NM],
    channels_seg=[LEDType.LED_450_NM],
)
adapter = MicroscopyCommandAdapter(
    image_brightness=image_brightness,
    segment_images=segment_images,
    save_images=save_images,
)

generation_timeout_s = 300
with StrategyGenerationService(
    pipeline,
    domain=domain,
    command_adapter=adapter,
    observation_provider=MicroscopyObservationProvider(),
) as service:
    future = service.submit(user_request, cfg)
    strategy = future.result(timeout=generation_timeout_s)

print(strategy.source)

INFO:httpx2:HTTP Request: POST https://robin-office-2.tail32bb7.ts.net/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://robin-office-2.tail32bb7.ts.net/v1/chat/completions "HTTP/1.1 200 OK"


initialise
    move_fov(target=first_fov)
    image(exposure=7ms, led=385nm)
    wait(duration=0.25s)

step
    if observation.step_count >= 12:
        terminate
    else:
        if observation.step_count < 3:
            if observation.current_fov_id == 0:
                image(exposure=11ms, led=450nm)
                wait(duration=0.5s)
                move_fov(target=next_fov)
            else:
                image(exposure=13ms, led=515nm)
                wait(duration=0.75s)
                move_fov(target=next_fov)
        else:
            if observation.step_count < 6:
                move_fov(target=next_fov)
                if observation.current_fov_id == 0:
                    image(exposure=17ms, led=565nm)
                    wait(duration=1s)
                else:
                    if observation.current_fov_id > 2:
                        image(exposure=19ms, led=645nm)
                        wait(duration=1.25s)
                    else:
                        

## Inspect the accepted result

In [24]:
verified = strategy.verified
print(f'Accepted after {len(verified.attempts)} candidate(s)')
print(f'Semantic verdict: {verified.verdict}')
print(f'Registered automaton command types: {[item.name for item in strategy.register_automaton_commands()]}')

Accepted after 1 candidate(s)
Semantic verdict: accepted=True issues=()
Registered automaton command types: ['WAIT', 'MOVE', 'IMAGE', 'STOP']


In [25]:
prompt = verified.accepted.generated.prompt
for index, message in enumerate(prompt.messages, start=1):
    print(f'\n--- Message {index}: {message.role.upper()} ---\n')
    print(message.content)


--- Message 1: SYSTEM ---

## Output contract

Generate one experimental strategy in the supplied strategy DSL. Return only the DSL source. Do not include Markdown fences, explanations, or any text before or after the strategy.

## DSL grammar

The generated strategy must follow this grammar exactly.

<grammar>
start: initialise step finalise

initialise: "initialise" _suite
step: "step" _suite
finalise: "finalise" _suite

_suite: _NL [_INDENT statement+ _DEDENT]

?statement: command_call | if_statement | control_action

command_call: IDENTIFIER "(" [argument_list] ")" _NL
argument_list: argument ("," argument)*
argument: IDENTIFIER "=" SOURCE_LITERAL

if_statement: "if" condition ":" if_body ["else" ":" else_body]
if_body: _block
else_body: _block

?condition: comparison | reference
comparison: reference COMPARISON_OPERATOR CONDITION_LITERAL
reference: REFERENCE_NAMESPACE "." IDENTIFIER

control_action: CONTROL_ACTION _NL

REFERENCE_NAMESPACE.2: "observation" | "error"
COMPARISON_OPE

## Convert the validated strategy into EvoMachine commands

The FOV mapping below simulates application-owned runtime configuration. The step preview simulates a callback at FOV `0`; no commands are executed.

In [26]:
fovs = {
    0: Coordinate(0, 0, 0),
    1: Coordinate(100, 0, 0),
}

initialise_commands = strategy.initialise(
    fovs=fovs,
    region_of_interests={fov_id: [] for fov_id in fovs},
    fov_processors={},
    dmd=None,
)
step_commands = strategy.callback(fov_id=0, data=[], errors=[])
finalise_commands = strategy.finalise()

In [27]:
def describe_command(command):
    details = command.command_args
    if command.command_type.name == 'IMAGE':
        metadata = details['frame_metadata']
        metadata_items = metadata if isinstance(metadata, list) else [metadata]
        details = {
            'frames': [
                {
                    'exposure_ms': item.exposure,
                    'leds': {led.name: brightness for led, brightness in (item.leds or {}).items()},
                }
                for item in metadata_items
            ],
            'segment': details['segment'],
            'save': details['save'],
        }
    return {
        'command_id': command.command_id,
        'command_type': command.command_type.name,
        'arguments': details,
    }


for lifecycle, commands in (
    ('initialise', initialise_commands),
    ('step', step_commands),
    ('finalise', finalise_commands),
):
    print(f'\n{lifecycle}:')
    if not commands:
        print('  (no commands)')
    for command in commands:
        print(f'  {describe_command(command)}')


initialise:
  {'command_id': 0, 'command_type': 'MOVE', 'arguments': 0}
  {'command_id': 1, 'command_type': 'IMAGE', 'arguments': {'frames': [{'exposure_ms': 7.0, 'leds': {'LED_385_NM': 10.0}}], 'segment': False, 'save': False}}
  {'command_id': 2, 'command_type': 'WAIT', 'arguments': {'duration': 0.25, 'set_live_mode': False, 'channel': <LEDType.LED_450_NM: 1>, 'brightness': 10}}

step:
  {'command_id': 3, 'command_type': 'IMAGE', 'arguments': {'frames': [{'exposure_ms': 11.0, 'leds': {'LED_450_NM': 10.0}}], 'segment': False, 'save': False}}
  {'command_id': 4, 'command_type': 'WAIT', 'arguments': {'duration': 0.5, 'set_live_mode': False, 'channel': <LEDType.LED_450_NM: 1>, 'brightness': 10}}
  {'command_id': 5, 'command_type': 'MOVE', 'arguments': -1}

finalise:
  {'command_id': 6, 'command_type': 'IMAGE', 'arguments': {'frames': [{'exposure_ms': 50.0, 'leds': {'LED_645_NM': 10.0}}], 'segment': False, 'save': False}}
  {'command_id': 7, 'command_type': 'WAIT', 'arguments': {'duratio

## Execute with the virtual automaton

This runs the generated strategy against EvoMachine's in-memory stage, camera, LEDs and DMD. The virtual automaton currently has one FOV, so `next_fov` wraps back to FOV `0`. A timeout and `finally` cleanup prevent an accidentally non-terminating generated strategy from leaving the worker running.

In [ ]:
from threading import Thread
from time import monotonic, sleep

from evomachine.gui.runtime import build_virtual_automaton


virtual_timeout_s = 100
inspection_channel = LEDType.LED_515_NM

In [30]:
virtual_automaton = build_virtual_automaton()
virtual_automaton.set_strategy(strategy)
execution_errors = []


def run_virtual_automaton():
    try:
        virtual_automaton.run()
    except Exception as error:
        execution_errors.append(error)


worker = Thread(
    target=run_virtual_automaton,
    name='autostrat-virtual-automaton',
    daemon=True,
)
deadline = monotonic() + virtual_timeout_s
completed_by_termination = False
run_summary = {}
latest_image = None

try:
    worker.start()
    virtual_automaton.start_strategy()
    while monotonic() < deadline:
        if execution_errors:
            break
        if virtual_automaton.stopped():
            completed_by_termination = True
            break
        sleep(0.01)

    run_summary = {
        'terminated': completed_by_termination,
        'completed_step_callbacks': strategy.callback_counter,
        'current_fov_id': virtual_automaton.get_fov_id(),
        'last_commands': [describe_command(command) for command in virtual_automaton.last_commands],
    }
    try:
        latest_image = virtual_automaton.get_frame(0, inspection_channel)
    except (KeyError, IndexError):
        latest_image = None
finally:
    virtual_automaton.shutdown()
    worker.join(timeout=5)

if execution_errors:
    raise RuntimeError('The virtual automaton failed.') from execution_errors[0]
if not completed_by_termination:
    raise TimeoutError(
        f'The generated strategy did not terminate within {virtual_timeout_s} seconds.'
    )

print(run_summary)
if latest_image is not None:
    print(
        {
            'latest_image_shape': latest_image.shape,
            'latest_image_min': float(latest_image.min()),
            'latest_image_max': float(latest_image.max()),
        }
    )

2026-08-27 20:36:15 - ERROR - evomachine.peripherals.dmd - Dmd.load_calibration_data: file /Users/liammetcalf/Code/workspace/evomachine/calibration_data/dmd/dmd_calibration_data_2025-08-14_v2.pkl not found.
2026-08-27 20:36:15 - DEBUG - evomachine.peripherals.camera - Camera.initialise: initialising Virtual Camera with force=False.
2026-08-27 20:36:15 - DEBUG - evomachine.peripherals.camera - Camera.set_exposure: setting Virtual Camera exposure to 200 ms.
2026-08-27 20:36:15 - DEBUG - evomachine.peripherals.camera - Camera.initialise: Virtual Camera initialised.
2026-08-27 20:36:15 - DEBUG - evomachine.peripherals.stage - Stage.initialise: initialising Virtual Stage with force=False.
2026-08-27 20:36:15 - DEBUG - evomachine.peripherals.stage - Stage.initialise: Virtual Stage initialised at (x=0.0, y=0.0, z=0.0, channel_id=0).
2026-08-27 20:36:15 - DEBUG - evomachine.peripherals.leds - LedManager.initialise: initialising LED Manager with force=False.
2026-08-27 20:36:15 - DEBUG - evomac

RuntimeError: The virtual automaton failed.